In [2]:
import json
import re
import random
import hashlib
import logging
from google.colab import files

# ── Logging ────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ── Configuration ──────────────────────────────────────────────────────────────
RANDOM_SEED = 42
INPUT_FILE  = "/content/master.jsonl"
OUTPUT_FILE = "/content/cleaned_master.jsonl"


# ══════════════════════════════════════════════════════════════════════════════
#  TEXT CLEANING UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

# Arabic tashkeel (diacritics) — purely phonetic marks, never change meaning
# in a school report context. Removing them reduces token count significantly.
# Unicode ranges covered:
#   U+0610–U+061A  : Arabic extended marks
#   U+064B–U+065F  : Core tashkeel (fatha, damma, kasra, tanwin, shadda, etc.)
#   U+0670         : Superscript alef
#   U+06D6–U+06DC  : Quranic annotation signs
#   U+06DF–U+06E4  : More Quranic marks
#   U+06E7–U+06E8  : Quranic marks
#   U+06EA–U+06ED  : Quranic marks
_TASHKEEL_RE = re.compile(
    r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06DC\u06DF-\u06E4\u06E7\u06E8\u06EA-\u06ED]"
)

# Hidden / zero-width / control characters that sometimes sneak into
# copy-pasted Arabic text from Word, PDFs, or web forms:
#   U+200B  Zero Width Space
#   U+200C  Zero Width Non-Joiner
#   U+200D  Zero Width Joiner
#   U+200E  Left-to-Right Mark
#   U+200F  Right-to-Left Mark
#   U+FEFF  Byte Order Mark (BOM)
#   U+00AD  Soft Hyphen
#   U+2060  Word Joiner
_HIDDEN_CHARS_RE = re.compile(r"[\u200B-\u200F\uFEFF\u00AD\u2060]")

# Any run of horizontal whitespace (spaces + tabs) longer than one → one space
_MULTI_SPACE_RE = re.compile(r"[ \t]{2,}")

# More than two consecutive newlines → exactly two (preserve paragraph breaks)
_MULTI_NEWLINE_RE = re.compile(r"\n{3,}")


def clean_text(text: str) -> str:
    """
    Apply lightweight Arabic text normalization.

    What this does
    ──────────────
    1. Remove tashkeel (diacritics)     — shrinks token count, no info lost
    2. Remove hidden/zero-width chars   — copy-paste artefacts from Word/PDF
    3. Normalize horizontal whitespace  — multiple spaces/tabs → single space
    4. Normalize excessive newlines     — 3+ blank lines → 2 (paragraph break)
    5. Strip leading/trailing whitespace per line and for the whole string

    What this does NOT do
    ─────────────────────
    • Does NOT touch bullet markers (•, -, *)
    • Does NOT reorder or merge lines
    • Does NOT alter punctuation (، . ؟ !)
    • Does NOT change Alef variants or any letter normalization
      (kept conservative to avoid unintended meaning changes)
    • Does NOT restructure or parse the content in any way
    """
    if not isinstance(text, str) or not text:
        return ""

    # 1. Strip tashkeel
    text = _TASHKEEL_RE.sub("", text)

    # 2. Remove hidden / zero-width characters
    text = _HIDDEN_CHARS_RE.sub("", text)

    # 3. Collapse multiple spaces/tabs within each line → single space
    text = _MULTI_SPACE_RE.sub(" ", text)

    # 4. Collapse 3+ consecutive newlines → 2 (keep intentional paragraph gaps)
    text = _MULTI_NEWLINE_RE.sub("\n\n", text)

    # 5. Strip each line individually, then strip the whole string
    lines = [line.strip() for line in text.splitlines()]
    text  = "\n".join(lines).strip()

    return text


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 1 — LOAD
# ══════════════════════════════════════════════════════════════════════════════

def load_jsonl(filepath: str) -> list[dict]:
    """
    Read a JSONL file line by line.
    Malformed JSON lines are logged and skipped — they never crash the pipeline.
    """
    records, bad = [], 0

    with open(filepath, encoding="utf-8") as fh:
        for line_no, raw in enumerate(fh, start=1):
            raw = raw.strip()
            if not raw:
                continue
            try:
                records.append(json.loads(raw))
            except json.JSONDecodeError as exc:
                log.warning("Line %d — bad JSON skipped: %s", line_no, exc)
                bad += 1

    log.info("Loaded  %d records  |  %d malformed lines skipped", len(records), bad)
    return records


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 2 — CLEAN
# ══════════════════════════════════════════════════════════════════════════════

def clean_records(records: list[dict]) -> list[dict]:
    """
    Apply clean_text() to every message role in every record.

    The function is role-agnostic on purpose: system, user, and assistant
    content all benefit from the same whitespace and diacritic normalization.
    The assistant's bullet-point structure is fully preserved because
    clean_text() only touches whitespace and invisible characters —
    it never alters the line structure or content semantics.

    Records missing a 'messages' list are dropped with a warning.
    """
    cleaned, dropped = [], 0

    for idx, obj in enumerate(records, start=1):
        messages = obj.get("messages")

        if not isinstance(messages, list) or not messages:
            log.warning("Record %d — missing or empty 'messages', dropped.", idx)
            dropped += 1
            continue

        # Clean every message's content regardless of role
        for msg in messages:
            if isinstance(msg.get("content"), str):
                msg["content"] = clean_text(msg["content"])

        obj["messages"] = messages
        cleaned.append(obj)

    log.info(
        "Cleaning             →  %d cleaned  |  %d dropped",
        len(cleaned), dropped,
    )
    return cleaned


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 3 — DEDUPLICATION
# ══════════════════════════════════════════════════════════════════════════════

def deduplicate(records: list[dict]) -> list[dict]:
    """
    Remove records whose user prompt has already been seen.

    Dedup key: MD5 hash of the stripped user message content.
    Hashing keeps the seen-set memory-efficient regardless of dataset size.

    Records with no user message are kept unconditionally (edge case).
    """
    seen, unique, dropped = set(), [], 0

    for obj in records:
        messages     = obj.get("messages", [])
        user_content = next(
            (m["content"] for m in messages if m.get("role") == "user"), None
        )

        if user_content is None:
            # No user turn found — keep but don't key on it
            unique.append(obj)
            continue

        key = hashlib.md5(user_content.strip().encode("utf-8")).hexdigest()

        if key in seen:
            dropped += 1
            continue

        seen.add(key)
        unique.append(obj)

    log.info(
        "Deduplication        →  %d unique   |  %d duplicates removed",
        len(unique), dropped,
    )
    return unique


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 4 — SHUFFLE
# ══════════════════════════════════════════════════════════════════════════════

def shuffle(records: list[dict]) -> list[dict]:
    """
    Randomly shuffle records in-place using a fixed seed.
    Fixed seed = reproducible order — re-running the script gives
    identical output, which is essential for experiment tracking.
    """
    random.seed(RANDOM_SEED)
    random.shuffle(records)
    log.info("Shuffled  %d records  (seed=%d).", len(records), RANDOM_SEED)
    return records


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 5 — SAVE
# ══════════════════════════════════════════════════════════════════════════════

def save_jsonl(records: list[dict], filepath: str) -> None:
    """
    Write records to a JSONL file.
    ensure_ascii=False → Arabic characters are stored as real Unicode,
    not escaped \\uXXXX sequences. This keeps the file human-readable and
    prevents token-count inflation during fine-tuning tokenisation.
    """
    with open(filepath, "w", encoding="utf-8") as fh:
        for rec in records:
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n")

    log.info("Saved    →  %s  (%d records)", filepath, len(records))


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main():
    SEP = "═" * 58

    log.info(SEP)
    log.info("  Elara Cleaning Pipeline  v5  —  START")
    log.info(SEP)

    # Stage 1 — Load
    raw      = load_jsonl(INPUT_FILE);       n0 = len(raw)
    log.info("─" * 58)

    # Stage 2 — Clean
    cleaned  = clean_records(raw);           n1 = len(cleaned)
    log.info("─" * 58)

    # Stage 3 — Deduplicate
    deduped  = deduplicate(cleaned);         n2 = len(deduped)
    log.info("─" * 58)

    # Stage 4 — Shuffle
    final    = shuffle(deduped)
    log.info("─" * 58)

    # Stage 5 — Save
    save_jsonl(final, OUTPUT_FILE)

    # ── Summary ────────────────────────────────────────────────────────────
    log.info(SEP)
    log.info("  PIPELINE SUMMARY")
    log.info(SEP)
    log.info("  Loaded              : %d", n0)
    log.info("  After cleaning      : %d  (-%d malformed)", n1, n0 - n1)
    log.info("  After dedup         : %d  (-%d duplicates)", n2, n1 - n2)
    log.info("  Final saved         : %d", len(final))
    log.info("  Overall retention   : %.1f%%", 100 * len(final) / n0 if n0 else 0)
    log.info(SEP)

    # Auto-download from Colab
    files.download(OUTPUT_FILE)


main()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>